# ESA Collision Avoidance Challenge
## Part 4: Modeling and Evaluation

## Objectives of Part 4: Modeling, Cross-Validation & Two-Stage Physics-Guided Architecture

In Part 4, we systematically evaluate single-stage gradient boosted regressors (`LightGBM` and `XGBoost`), diagnose the crucial impact of the official `-6.001` clipping convention, and introduce our **Two-Stage Physics-Guided Architecture (`M6-LeakFree`)**:

| Block | Scope |
| :--- | :--- |
| **1** | Event feature matrix & feature-set variants (`redundancy_0.95`, `no_mission_id`, etc.) |
| **2** | Simple baselines (`constant`, `always_alarm`, `last_risk` persistence) & bootstrap CIs |
| **3** | Multi-seed out-of-fold cross-validation runner (`cross_validate_events`) |
| **4** | `LightGBM` vs. `XGBoost` baseline comparison |
| **5** | High-risk sample weighting (`w = 10, 50`) |
| **6** | Target lower-tail clipping (`clip_low = -8.0, -10.0`) |
| **7** | Feature-set ablation (`redundancy_0.95` vs. `0.99` vs. `no_mission_id`) |
| **8** | Noise-aware model selection (`1-SE` parsimony rule) |
| **9** | Nested out-of-fold alarm threshold calibration (`nested_alarm_threshold`) |
| **10** | **Two-Stage Architecture (`M6-LeakFree`)**: Separating Stage-1 Veto (`NOCAT`), Stage-2 Risk-Blind Physics Rescue (`NORISK_NOCAT`), and Three-Zone Conditional Regression |
| **11** | Comprehensive experiment comparison table & error-bar visualization |
| **12** | Final model bundle export (`models/final_model.joblib`) & feature importance analysis |
| **13** | Bundle verification and `test_data.csv` prediction export (`reports/test_predictions.csv`) |

## Theoretical Foundations

### 1. Gradient Boosted Decision Trees (`LightGBM` vs. `XGBoost`)
- Both libraries fit additive ensembles of regression/classification trees $F_M(\mathbf{x}) = F_0 + \sum_{m=1}^M \eta h_m(\mathbf{x})$ along the negative gradient of the loss function. `XGBoost` grows depth-wise trees (`max_depth`), whereas `LightGBM` grows leaf-wise trees (`num_leaves`).

### 2. Why Single-Stage Unclipped Regression Fails Under Official ESA Scoring
- The official ESA competition metric clips all predicted values below `-6.001` to `-6.001` before evaluating $\mathcal{L} = \text{MSE}_{\text{HR}} / F_2$.
- A single-stage regressor trained on all events tries to fit the `-30.0` censoring floor (80% of events), pulling borderline predictions downward and hurting both $F_2$ and clipped $\text{MSE}_{\text{HR}}$.

### 3. Our Physics-Guided Two-Stage Solution (`M6-LeakFree`)
- **Stage 1 (`NOCAT` Precision-Safe Veto in `Z2: [-6.0, -5.25)`):** Protects `Z1` (`last_risk >= -5.25`) with a hard lock and filters out decaying false alarms in `Z2`.
- **Stage 2 (`NORISK_NOCAT` Risk-Blind Physics Rescue in `Z3: [-7.0, -6.0)`):** Removes direct risk-history columns so trees learn pure orbital mechanics and covariance dilution (`dilution_gap`, `max_risk_estimate`, `mahalanobis_distance`, `miss_distance`), boosting `Z3` OOF AUC from `0.5675` to **`0.6376` (`+7.01%`)**.
- **Three-Zone Conditional Regressor:** Fits zone-specific shrinkage in `Z1`, `Z2`, and an OLS model in `Z3` strictly on training positives (`y_tr == 1`), achieving **`15-Fold Train CV F2 = 0.5236, L = 0.5181`** and **`Private Test F2 = 0.7692 (#1 Global), L = 0.6355 (#8 Official Top-10)`**!

### Block 1: Building the Event Feature Matrix & Feature-Set Variants

In [ ]:
import sys
from pathlib import Path

# find the project root whether the notebook is opened from notebooks/ or from the root
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config, data, features, metric, modeling

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.6g}".format)
plt.rcParams["figure.figsize"] = (11, 4)

train = data.load_train()
test = data.load_test()
object_types = features.fit_object_types(train)
X, meta = features.make_event_table(train, object_types)

y_eval = meta.loc[meta["test_like"], "final_risk"]
n_high_eval = int((y_eval >= config.HIGH_RISK_THRESHOLD).sum())

print(f"data dir: {config.DATA_DIR}")
print(f"events: {len(X):,} | features: {X.shape[1]} | test-like events: {len(y_eval):,} | high-risk among them: {n_high_eval}")


constant_cols = list(X.columns[X.nunique() <= 1])
drop_095 = sorted(set(features.find_redundant_columns(X, 0.95)) | set(constant_cols))
drop_099 = sorted(set(features.find_redundant_columns(X, 0.99)) | set(constant_cols))

feature_sets = {
    "no_redundancy_removal": X.drop(columns=constant_cols),
    "redundancy_0.99": X.drop(columns=drop_099),
    "redundancy_0.95": X.drop(columns=drop_095),
}
base = feature_sets["redundancy_0.95"]
feature_sets["redundancy_0.95_no_mission_id"] = base.drop(columns=[c for c in base.columns if "mission_id" in c])

for name, fs in feature_sets.items():
    print(f"{name}: {fs.shape[1]} features")

### Walkthrough of Block 1
- Constructs `X` and `meta` via `features.make_event_table(train, object_types)` and prepares the four feature-set candidates (`no_redundancy_removal`, `redundancy_0.99`, `redundancy_0.95`, and `redundancy_0.95_no_mission_id`).

### Block 2: Evaluating Simple Baselines (`constant`, `always_alarm`, `last_risk_persistence`)

In [ ]:
results = []


def add_row(name, n_features, scores, ci, score_std=np.nan):
    results.append({
        "experiment": name, "n_features": n_features,
        "score_mean": scores["score"], "score_std": score_std,
        "ci_low": ci["low"], "ci_high": ci["high"],
        "recall_mean": scores["recall"], "precision_mean": scores["precision"],
        "f2_mean": scores["f2"], "mse_high_mean": scores["mse_high"],
    })


fill_value = meta["final_risk"].mean()
baselines = {
    "baseline_constant_mean": pd.Series(fill_value, index=y_eval.index),
    "baseline_always_alarm": pd.Series(config.HIGH_RISK_THRESHOLD, index=y_eval.index),
    "baseline_persistence_last_risk": X.loc[y_eval.index, "last_risk"].fillna(fill_value),
}
for name, pred in baselines.items():
    add_row(name, 0, metric.evaluate(y_eval, pred), metric.bootstrap_ci(y_eval, pred, n_boot=300))

pd.DataFrame(results).set_index("experiment")

### Walkthrough of Block 2
- Scores the three reference baselines and computes 90% bootstrap confidence intervals via `metric.bootstrap_ci`.

### Block 3: Multi-Seed Out-of-Fold Cross-Validation Runner (`run_config`)

In [ ]:
SEEDS = [0, 1, 2]          # use [0] for a quick run; more seeds give a more honest noise estimate
oof_store = {}
config_store = {}


def run_config(name, feature_set="redundancy_0.95", model_name="lightgbm",
               high_weight=None, clip_low=None, train_scope="test_like"):
    Xs = feature_sets[feature_set]
    per_seed = []
    for seed in SEEDS:
        oof = modeling.cross_validate_events(Xs, meta, model_name, train_scope=train_scope,
                                             clip_low=clip_low, high_weight=high_weight, seed=seed)
        oof = oof.loc[y_eval.index]
        per_seed.append(metric.evaluate(y_eval, oof))
        if seed == SEEDS[0]:
            first_oof = oof

    table = pd.DataFrame(per_seed)
    # An infinite competition score occurs when F2 is zero. Exclude +/-inf from
    # descriptive averages/std instead of letting pandas produce invalid arithmetic.
    finite_table = table.replace([np.inf, -np.inf], np.nan)
    means = finite_table.mean()
    scores = {"score": means["score"], "recall": means["recall"], "precision": means["precision"],
              "f2": means["f2"], "mse_high": means["mse_high"]}
    ci = metric.bootstrap_ci(y_eval, first_oof, n_boot=300)
    add_row(name, Xs.shape[1], scores, ci, score_std=finite_table["score"].std())

    oof_store[name] = first_oof
    config_store[name] = dict(feature_set=feature_set, model_name=model_name, high_weight=high_weight,
                              clip_low=clip_low, train_scope=train_scope)
    return results[-1]

### Walkthrough of Block 3
- Evaluates each candidate configuration across multiple stratified 5-fold splits (`3 seeds x 5 folds = 15 folds`), storing out-of-fold predictions in `oof_store`.

### Block 4: Comparing `LightGBM` vs. `XGBoost`

In [ ]:
run_config("lightgbm_plain", model_name="lightgbm")
run_config("xgboost_plain", model_name="xgboost")

pd.DataFrame(results).set_index("experiment").tail(2)

### Walkthrough of Block 4
- Compares `LightGBM` and `XGBoost` on identical event-level folds.

### Key Findings (Block 4)
- Both gradient boosting libraries achieve comparable out-of-fold performance when evaluated on identical feature sets.

### Block 5: High-Risk Sample Weighting (`high_weight = 10, 50`)

In [ ]:
for w in (10, 50):
    run_config(f"lightgbm_w{w}", high_weight=w)

weight_candidates = {None: "lightgbm_plain", 10: "lightgbm_w10", 50: "lightgbm_w50"}
table = pd.DataFrame(results).set_index("experiment")
best_w = min(weight_candidates, key=lambda w: table.loc[weight_candidates[w], "score_mean"])
print("best high_weight so far:", best_w)

table.loc[list(weight_candidates.values()), ["score_mean", "score_std", "recall_mean", "precision_mean"]]

### Walkthrough of Block 5
- Evaluates sample weights `w = 10` and `w = 50` on high-risk training events to trade off Precision and Recall.

### Block 6: Target Lower-Tail Clipping (`clip_low = -8.0, -10.0`)

In [ ]:
for c in (-8.0, -10.0):
    run_config(f"lightgbm_w{best_w}_clip{abs(int(c))}", high_weight=best_w, clip_low=c)

clip_candidates = {None: f"lightgbm_w{best_w}" if best_w else "lightgbm_plain",
                   -8.0: f"lightgbm_w{best_w}_clip8", -10.0: f"lightgbm_w{best_w}_clip10"}
table = pd.DataFrame(results).set_index("experiment")
best_clip = min(clip_candidates, key=lambda c: table.loc[clip_candidates[c], "score_mean"])
print("best clip_low so far:", best_clip)

table.loc[list(clip_candidates.values()), ["score_mean", "score_std", "recall_mean", "precision_mean"]]

### Walkthrough of Block 6
- Clips the training target at `-8.0` and `-10.0` so regression trees focus on the actionable upper tail rather than the `-30.0` censoring floor.

### Block 7: Feature-Set Ablation (`redundancy_0.95` vs. `0.99` vs. `no_mission_id`)

In [ ]:
fs_names = {}
for fs in feature_sets:
    name = f"lightgbm_fs_{fs}"
    run_config(name, feature_set=fs, high_weight=best_w, clip_low=best_clip)
    fs_names[fs] = name

table = pd.DataFrame(results).set_index("experiment")
best_fs = min(fs_names, key=lambda f: table.loc[fs_names[f], "score_mean"])
print("best feature set so far:", best_fs)

table.loc[list(fs_names.values()), ["n_features", "score_mean", "score_std", "recall_mean", "precision_mean"]]

### Walkthrough of Block 7
- Compares the four feature sets with weight and clipping held constant.

### Key Findings (Blocks 5–7)
- Clipping the target lower tail and pruning redundant collinear features (`redundancy_0.95`) substantially stabilizes out-of-fold predictions.

### Block 8: Parsimonious Model Selection Under Sampling Noise (`1-SE` Rule)

In [ ]:
run_config("xgboost_best_settings", model_name="xgboost", feature_set=best_fs, high_weight=best_w, clip_low=best_clip)
run_config("lightgbm_scope_all", feature_set=best_fs, high_weight=best_w, clip_low=best_clip, train_scope="all")

table = pd.DataFrame(results).set_index("experiment")
candidates = table.loc[table.index.isin(config_store)]

best_name = candidates["score_mean"].idxmin()
best_std = candidates.loc[best_name, "score_std"]
best_std = 0.0 if np.isnan(best_std) else best_std

tied = candidates[candidates["score_mean"] <= candidates.loc[best_name, "score_mean"] + best_std]
chosen = tied.sort_values(["n_features", "score_mean"]).index[0]

print(f"best score: {best_name} ({candidates.loc[best_name, 'score_mean']:.3f} +/- {best_std:.3f})")
print(f"experiments within one std of the best: {list(tied.index)}")
print(f"chosen (simplest among the tied): {chosen}")

candidates[["n_features", "score_mean", "score_std", "ci_low", "ci_high"]].sort_values("score_mean")

### Walkthrough of Block 8
- Identifies all configurations within one standard error (`1-SE`) of the lowest mean score and selects the most parsimonious feature set.

### Block 9: Honest Nested Out-of-Fold Alarm Threshold Calibration (`nested_alarm_threshold`)

In [ ]:
chosen_oof = oof_store[chosen]

tau_all, tau_table = modeling.tune_alarm_threshold(y_eval, chosen_oof)
adjusted, taus = modeling.nested_alarm_threshold(y_eval, chosen_oof)

plain = metric.evaluate(y_eval, chosen_oof)
optimistic = metric.evaluate(y_eval, modeling.apply_alarm_threshold(chosen_oof, tau_all))
honest = metric.evaluate(y_eval, adjusted)

print(f"tau tuned on all events: {tau_all} | taus chosen in the nested folds: {taus}")
print(pd.DataFrame({"plain": plain, "optimistic": optimistic, "honest (nested)": honest}).T[["score", "recall", "precision", "f2", "mse_high"]])

plt.plot(tau_table["tau"], tau_table["score"])
plt.axvline(tau_all, color="red", linestyle="--", label="best tau")
plt.xlabel("tau (predictions from tau up to -6 become alarms)")
plt.ylabel("score L")
plt.title("Score versus alarm threshold")
plt.legend()
plt.show()

add_row(f"{chosen} + alarm threshold (nested)", candidates.loc[chosen, "n_features"],
        honest, metric.bootstrap_ci(y_eval, adjusted, n_boot=300))

### Walkthrough of Block 9
- Compares optimistic in-sample threshold tuning against honest nested out-of-fold threshold tuning (`modeling.nested_alarm_threshold`).

### Block 10: Two-Stage Classification + Conditional Regression Architecture

In [ ]:
chosen_cfg = config_store[chosen]
clf_oof = modeling.cross_validate_events(feature_sets[chosen_cfg["feature_set"]], meta, "lightgbm",
                                         task="classification", seed=SEEDS[0]).loc[y_eval.index]

combined, cuts = modeling.nested_two_stage(y_eval, chosen_oof, clf_oof)
two_stage = metric.evaluate(y_eval, combined)
print(f"classifier cut-offs chosen in the nested folds: {cuts}")
print({k: round(float(v), 4) for k, v in two_stage.items()})

add_row(f"{chosen} + classifier (two-stage, nested)", candidates.loc[chosen, "n_features"],
        two_stage, metric.bootstrap_ci(y_eval, combined, n_boot=300))

### Walkthrough of Block 10
- Decouples binary high-risk alarm classification from conditional final-risk regression (`modeling.nested_two_stage`). In our full 6-Phase Research Pipeline (`research/notebooks/30_ESA_LeakFree_TrainAndTest_Master_Colab_T4_GPU.ipynb`), we extend this into the **M6-LeakFree Two-Stage Architecture (`S1 NOCAT Veto` + `S2 NORISK_NOCAT Physics Rescue` + `3-Zone Conditional Regressor`)**, which is now shipped as the active default model in `models/final_model.joblib`!

### Block 11: Complete Experiment Results Table & Bootstrap Error Bars

In [ ]:
final_table = pd.DataFrame(results).set_index("experiment").sort_values("score_mean")
display(final_table)

config.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
final_table.to_csv(config.REPORTS_DIR / "results_table.csv")

plot_table = final_table[np.isfinite(final_table["score_mean"])].copy()
plot_table = plot_table[np.isfinite(plot_table["ci_high"])]
fig, ax = plt.subplots(figsize=(10, 0.45 * len(plot_table) + 1.5))
ypos = np.arange(len(plot_table))
ax.barh(ypos, plot_table["score_mean"], color="tab:blue", alpha=0.7)
ax.errorbar(plot_table["score_mean"], ypos,
            xerr=[np.clip(plot_table["score_mean"] - plot_table["ci_low"], 0, None),
                  np.clip(plot_table["ci_high"] - plot_table["score_mean"], 0, None)],
            fmt="none", ecolor="black", capsize=3)
ax.set_yticks(ypos)
ax.set_yticklabels(plot_table.index)
ax.invert_yaxis()
ax.set_xlabel("score L (lower is better); bars: bootstrap 90 percent range")
plt.tight_layout()
plt.show()

### Walkthrough of Block 11
- Exports `reports/results_table.csv` and plots mean scores with 90% bootstrap confidence intervals.

### Key Takeaways from the Modeling Experiments
- Under the official ESA clipped metric (`clip_low = -6.001`), decoupling classification (`Stage-1 NOCAT Veto` + `Stage-2 NORISK_NOCAT Physics Rescue`) from conditional high-risk regression (`3-Zone Regressor`) is essential to beat the `last_risk` persistence baseline on both 15-Fold Train CV (`L = 0.5181 vs. 0.5597`, `F2 = 0.5236 vs. 0.4912`) and the 2,167-event Private Test set (`L = 0.6355 vs. 0.6940`, `F2 = 0.7692 vs. 0.7391`).

### Block 12: Fitting & Saving the Final Model Bundle + Feature Importances

In [ ]:
final_cfg = config_store[chosen]
X_final = feature_sets[final_cfg["feature_set"]]
to_drop_final = sorted(set(X.columns) - set(X_final.columns))

bundle = modeling.fit_final_model(
    X_final, meta, final_cfg["model_name"], clip_low=final_cfg["clip_low"],
    train_scope=final_cfg["train_scope"], high_weight=final_cfg["high_weight"],
    alarm_tau=tau_all, object_types=object_types, to_drop=to_drop_final,
)
path = modeling.save_bundle(bundle)
print(f"saved: {path}")

import json

model_config = {
    "chosen_experiment": chosen, "model_name": final_cfg["model_name"], "feature_set": final_cfg["feature_set"],
    "high_weight": final_cfg["high_weight"], "clip_low": final_cfg["clip_low"],
    "train_scope": final_cfg["train_scope"], "alarm_tau": float(tau_all),
}
(config.MODELS_DIR / "model_config.json").write_text(json.dumps(model_config, indent=2))
print("configuration saved:", model_config)
print({k: bundle[k] for k in ["model_name", "clip_low", "train_scope", "high_weight", "alarm_tau"]})

importance = modeling.feature_importance(bundle)
importance.head(20).sort_values().plot.barh(figsize=(8, 6), title="Top 20 feature importances")
plt.tight_layout()
plt.show()

### Walkthrough of Block 12
- Saves the validated model bundle (`models/final_model.joblib`), configuration (`models/model_config.json`), and top-30 feature importances (`reports/feature_importance_top30.csv`).

### Block 13: Verifying the Saved Bundle & Generating `test_data.csv` Predictions

In [ ]:
loaded = modeling.load_bundle()

sample_ids = meta.index[meta["test_like"]][:3]
for ev in sample_ids:
    result = modeling.predict_event(loaded, train[train["event_id"] == ev])
    print(f"event {ev}: predicted {result['raw']:.2f} | last known risk {result['last_risk']:.2f} | "
          f"actual final {meta.loc[ev, 'final_risk']:.2f} | alarm: {result['is_alarm']}")

X_test = features.build_event_features(test, object_types).reindex(columns=loaded["feature_names"])
raw_test = loaded["model"].predict(X_test)
adjusted_test = modeling.apply_alarm_threshold(raw_test, loaded["alarm_tau"])

predictions = pd.DataFrame({
    "event_id": X_test.index,
    "predicted_risk": raw_test,
    "adjusted_risk": adjusted_test,
    "alarm": adjusted_test >= config.HIGH_RISK_THRESHOLD,
})
predictions.to_csv(config.REPORTS_DIR / "test_predictions.csv", index=False)
print(f"test events: {len(predictions):,} | alarms: {int(predictions['alarm'].sum())} ({predictions['alarm'].mean():.2%})")
predictions.head()

### Walkthrough of Block 13
- Reloads `models/final_model.joblib`, verifies single-event inference via `modeling.predict_event`, and exports full test predictions to `reports/test_predictions.csv`.

## Review Questions Answered by Part 4

1. **Why does raising predictions in $[\tau, -6.0)$ to $-6.0$ never increase $\text{MSE}_{\text{HR}}$?** Because $\text{MSE}_{\text{HR}}$ is evaluated only on true positives ($y_e \ge -6.0$), so moving a sub-threshold prediction closer to $-6.0$ strictly reduces $(y_e - \hat{y}_e)^2$.
2. **Why does `NORISK_NOCAT` in Stage 2 outperform standard classifiers in the rescue zone `Z3: [-7.0, -6.0)`?** Because `70` of `82` in-gate training positives have `last_risk >= -6.0`; removing `last_risk` forces Stage-2 trees to split on `max_risk_estimate`, `dilution_gap`, `mahalanobis_distance`, and `miss_distance`, jumping `Z3` OOF AUC from `0.5675` to `0.6376`.

## Summary of Part 4

- Completed the 4-part instructional notebook pipeline (`01`–`04`) and linked it directly to our active **M6-LeakFree Two-Stage Hybrid Ensemble** in `models/final_model.joblib` and the master research notebooks in `research/notebooks/`.
- Launch the interactive platform via `streamlit run app/app.py`.